In [1]:
# Imports 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import collections
import os

warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    cross_val_score, GridSearchCV, RandomizedSearchCV
)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, classification_report, roc_curve
)
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss, cohen_kappa_score, make_scorer

# Use imblearn Pipeline so SMOTE runs inside each CV fold
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

import joblib

RANDOM_STATE = 42
os.makedirs('outputs', exist_ok=True)
print('All imports successful.')

All imports successful.


In [2]:
# Load cleaned diabetes dataset
diabetes_df = pd.read_csv('../data/diabetes/diabetes_cleaned.csv')

print('Dataset loaded successfully')
print('Shape:', diabetes_df.shape)
print('\nColumn names:')
print(diabetes_df.columns.tolist())
print(diabetes_df.head())

Dataset loaded successfully
Shape: (768, 9)

Column names:
['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']
   Pregnancies   Glucose  BloodPressure  SkinThickness   Insulin       BMI  \
0     0.639947  0.866045      -0.031990       0.670643 -0.181541  0.166619   
1    -0.844885 -1.205066      -0.528319      -0.012301 -0.181541 -0.852200   
2     1.233880  2.016662      -0.693761      -0.012301 -0.181541 -1.332500   
3    -0.844885 -1.073567      -0.528319      -0.695245 -0.540642 -0.633881   
4    -1.141852  0.504422      -2.679076       0.670643  0.316566  1.549303   

   DiabetesPedigreeFunction       Age  Outcome  
0                  0.468492  1.425995        1  
1                 -0.365061 -0.190672        0  
2                  0.604397 -0.105584        1  
3                 -0.920763 -1.041549        0  
4                  5.484909 -0.020496        1  


In [3]:
# Quick sanity check — missing values and data types
print('Missing values:')
print(diabetes_df.isnull().sum())
print('\nData types:')
print(diabetes_df.dtypes)
print('\nTarget distribution (Outcome):')
print(diabetes_df['Outcome'].value_counts())

Missing values:
Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

Data types:
Pregnancies                 float64
Glucose                     float64
BloodPressure               float64
SkinThickness               float64
Insulin                     float64
BMI                         float64
DiabetesPedigreeFunction    float64
Age                         float64
Outcome                       int64
dtype: object

Target distribution (Outcome):
Outcome
0    500
1    268
Name: count, dtype: int64


In [4]:
# Separate features (X) and target (y)
X = diabetes_df.drop(columns=['Outcome'])
y = diabetes_df['Outcome']   # 0 = No Diabetes, 1 = Diabetes

print('Features shape:', X.shape)
print('Target shape:  ', y.shape)
print('\nClass distribution:')
print(y.value_counts())
print(f'\nClass imbalance ratio — No Diabetes : Diabetes = '
      f'{y.value_counts()[0]}:{y.value_counts()[1]}')

Features shape: (768, 8)
Target shape:   (768,)

Class distribution:
Outcome
0    500
1    268
Name: count, dtype: int64

Class imbalance ratio — No Diabetes : Diabetes = 500:268


In [5]:
# Train-test split (stratified to maintain class distribution)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f'Train : {X_train.shape[0]} samples')
print(f'Test  : {X_test.shape[0]} samples')
print('\nTrain class distribution:')
print(y_train.value_counts())
print('\nTest class distribution:')
print(y_test.value_counts())

Train : 614 samples
Test  : 154 samples

Train class distribution:
Outcome
0    400
1    214
Name: count, dtype: int64

Test class distribution:
Outcome
0    100
1     54
Name: count, dtype: int64


In [6]:
# Scaling (fit on train only)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform
X_test_scaled  = scaler.transform(X_test)         # transform only — no leakage

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=X.columns)

print('Scaling done (fit on train only — no leakage).')
print('\nTrain scaled stats (mean ≈ 0, std ≈ 1):')
print(X_train_scaled.describe().loc[['mean', 'std']].round(3))

Scaling done (fit on train only — no leakage).

Train scaled stats (mean ≈ 0, std ≈ 1):
      Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin    BMI  \
mean       -0.000    0.000         -0.000          0.000   -0.000 -0.000   
std         1.001    1.001          1.001          1.001    1.001  1.001   

      DiabetesPedigreeFunction    Age  
mean                    -0.000 -0.000  
std                      1.001  1.001  


In [7]:
# CV Setup
# SMOTE is NOT applied here — it lives inside each pipeline below
skf          = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
kappa_scorer = make_scorer(cohen_kappa_score)

def cv_report(pipeline, X, y, label):
    """
    Honest 5-fold CV: SMOTE runs inside each fold via ImbPipeline.
    No synthetic samples ever leak into the validation fold.
    """
    acc   = cross_val_score(pipeline, X, y, cv=skf, scoring='accuracy')
    roc   = cross_val_score(pipeline, X, y, cv=skf, scoring='roc_auc')
    kappa = cross_val_score(pipeline, X, y, cv=skf, scoring=kappa_scorer)
    print(f'\n[{label}]  5-Fold CV (SMOTE inside fold)')
    print(f'  Accuracy : {acc.mean():.4f}  ±  {acc.std():.4f}')
    print(f'  ROC-AUC  : {roc.mean():.4f}  ±  {roc.std():.4f}')
    print(f'  Kappa    : {kappa.mean():.4f}  ±  {kappa.std():.4f}  ← KEY')
    return {'Accuracy': acc.mean(), 'ROC-AUC': roc.mean(), 'Kappa': kappa.mean()}

cv_results = {}
print('CV setup ready.')

CV setup ready.


# Model 1: Logistic Regression

In [8]:
# Logistic Regression pipeline
lr_pipeline = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('classifier', LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=RANDOM_STATE
    ))
])

# Parameter grid
lr_param_grid = {
    'classifier__C': [0.01, 0.1, 1, 10, 100],
    'classifier__penalty': ['l1', 'l2'],
    'classifier__solver': ['liblinear']
}

# GridSearchCV
lr_grid = GridSearchCV(
    estimator=lr_pipeline,
    param_grid=lr_param_grid,
    cv=skf,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=0
)

# Fit on raw training data
lr_grid.fit(X_train, y_train)

print('Best LR params :', lr_grid.best_params_)
print('Best CV ROC-AUC:', round(lr_grid.best_score_, 4))

lr_best = lr_grid.best_estimator_
cv_results['Logistic Regression'] = cv_report(
    lr_best, X_train, y_train, 'Logistic Regression'
)

Best LR params : {'classifier__C': 0.1, 'classifier__penalty': 'l2', 'classifier__solver': 'liblinear'}
Best CV ROC-AUC: 0.8443

[Logistic Regression]  5-Fold CV (SMOTE inside fold)
  Accuracy : 0.7671  ±  0.0072
  ROC-AUC  : 0.8443  ±  0.0157
  Kappa    : 0.5007  ±  0.0125  ← KEY
